<a href="https://colab.research.google.com/github/Shannonfireball/bookRecommendation/blob/main/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-03-16 11:54:08--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 172.67.70.149, 104.26.3.33, 104.26.2.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|172.67.70.149|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  86.5MB/s    in 0.3s    

2026-03-16 11:54:09 (86.5 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [ ]:
# add your code here - consider creating a new cell for each section of code
print("df_books.columns",df_books.columns)
print("df_books.columns",df_ratings.columns)
merged_df = pd.merge( df_ratings, df_books, on="isbn" )

userCount = merged_df['user'].value_counts()
bookCount = merged_df['isbn'].value_counts()


merged_df = merged_df[ ~merged_df['user'].isin( userCount[ userCount < 200 ].index ) ]
merged_df = merged_df[ ~merged_df['isbn'].isin( bookCount[ bookCount < 100 ].index ) ]

pivot_df = merged_df.pivot_table( index='title', columns='user', values='rating')
pivot_df.fillna(0, inplace=True)

itemMatrix = csr_matrix(pivot_df.values)

In [ ]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):
  model = NearestNeighbors(metric='cosine', algorithm='brute')
  model.fit(itemMatrix)
  distances, indices = model.kneighbors(pivot_df.iloc[0,:].values.reshape(1,-1), n_neighbors=6)

  similarItems = [ [pivot_df.index[indices.flatten()[i]], distances.flatten()[i].item()] for i in range(1, len(distances.flatten())) ]
  # for i in range(len(distances.flatten())):
  #   print(f"{i}: {pivot_df.index[indices.flatten()[i]]} (Distance: {distances.flatten()[i]})")
  #   similarItems[1].append([ pivot_df.index[indices.flatten()[i]], distances.flatten()[i] ])
  similarItems.reverse()
  output = [book, similarItems]
  return output

books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print("books",books)

In [31]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [['American Psycho (Vintage Contemporaries)', 0.7725098729133606], ["The Hitchhiker's Guide to the Galaxy", 0.7693151831626892], ["Slaughterhouse Five or the Children's Crusade: A Duty Dance With Death", 0.7689031958580017], ['The Catcher in the Rye', 0.7566767930984497], ["The Handmaid's Tale", 0.7451601028442383]]]
You haven't passed yet. Keep trying!
